In [50]:
import pandas as pd
import numpy as np


In [51]:
# Carregamento dos dados brutos do formulário
df = pd.read_csv("F-AHP_Teste.csv", delimiter=';')
pd.set_option("display.float_format", "{:.3f}".format)
display(df)


,Name,Email,Area Atuacao,Instituicao,Intensidade Saaty A Vs B,Intensidade Saaty A Vs C,Intensidade Saaty A Vs D,Intensidade Saaty A Vs E,Intensidade Saaty A Vs F,Intensidade Saaty B Vs C,...,Mais Importante B Vs E,Mais Importante B Vs F,Mais Importante C Vs D,Mais Importante C Vs E,Mais Importante C Vs F,Mais Importante D Vs E,Mais Importante D Vs F,Mais Importante E Vs F,Message,Subject
0,Dr. Marcos Vinícius Lima,marcos.lima@infra-energy.gov.br,"Logística, Transporte Dutoviário e Infraestrut...",Agência Nacional de Energia / ANP,Igual importância,Moderadamente superior,Superior,Muito Superior,Extremamente superior,Igual importância,...,B. Reações Química,F. Logística e condições do sítio,C. Geomecânica da Formação,E. Rocha selante secundária,C. Geomecânica da Formação,E. Rocha selante secundária,D. Características das Intercamadas,F. Logística e condições do sítio,Submissão de teste 5/5: Logística e proximidad...,Avaliação Fuzzy-AHP - Dr. Marcos Vinícius Lima
1,Dra. Beatriz Albuquerque,beatriz.albuquerque@geoquimica-co2.com,Geoquímica e Reatividade Fluido-Rocha em Ambie...,Consultoria GeoAmbiente,Muito Superior,Extremamente superior,Igual importância,Moderadamente superior,Superior,Muito Superior,...,E. Rocha selante secundária,B. Reações Química,D. Características das Intercamadas,C. Geomecânica da Formação,F. Logística e condições do sítio,D. Características das Intercamadas,F. Logística e condições do sítio,E. Rocha selante secundária,Submissão de teste 4/5: Reações químicas e dis...,Avaliação Fuzzy-AHP - Dra. Beatriz Albuquerque
2,Prof. Dr. Roberto Guimarães,roberto.guimaraes@fem.unicamp.br,Geomecânica de Rochas Salinas e Fluência de Ev...,UNICAMP,Moderadamente superior,Superior,Muito Superior,Extremamente superior,Igual importância,Moderadamente superior,...,B. Reações Química,F. Logística e condições do sítio,C. Geomecânica da Formação,E. Rocha selante secundária,C. Geomecânica da Formação,E. Rocha selante secundária,D. Características das Intercamadas,F. Logística e condições do sítio,Submissão de teste 3/5: Foco na estabilidade e...,Avaliação Fuzzy-AHP - Prof. Dr. Roberto Guimarães
3,Dr. Carlos Eduardo Mendes,carlos.mendes@ccus-brasil.org,Engenharia de Reservatórios e Armazenamento Ge...,Instituto Nacional de CCUS,Extremamente superior,Igual importância,Moderadamente superior,Superior,Muito Superior,Extremamente superior,...,E. Rocha selante secundária,B. Reações Química,D. Características das Intercamadas,C. Geomecânica da Formação,F. Logística e condições do sítio,D. Características das Intercamadas,F. Logística e condições do sítio,E. Rocha selante secundária,Submissão de teste 2/5: Ênfase em geomecânica ...,Avaliação Fuzzy-AHP - Dr. Carlos Eduardo Mendes
4,Dra. Helena Vasconcelos,helena.vasconcelos@labgeo-ufrj.br,Geologia Estrutural e Estratigrafia de Bacias ...,UFRJ / CENPES,Superior,Muito Superior,Extremamente superior,Igual importância,Moderadamente superior,Superior,...,B. Reações Química,F. Logística e condições do sítio,C. Geomecânica da Formação,E. Rocha selante secundária,C. Geomecânica da Formação,E. Rocha selante secundária,D. Características das Intercamadas,F. Logística e condições do sítio,Submissão de teste 1/5: Priorização de formaçã...,Avaliação Fuzzy-AHP - Dra. Helena Vasconcelos


In [52]:
from decimal import Decimal, ROUND_HALF_UP

# Definição da Escala Fuzzy Triangular de Saaty (l, m, u)
scale = {
    "Igual importância": [1.0, 1.0, 1.0],
    "Moderadamente superior": [2.0, 3.0, 4.0],
    "Superior": [4.0, 5.0, 6.0],
    "Muito Superior": [6.0, 7.0, 8.0],
    "Extremamente superior": [9.0, 9.0, 9.0],
}

# Identificadores dos 6 macrocritérios avaliados
criterios = ["A", "B", "C", "D", "E", "F"]
n = len(criterios)

# ==============================================================================
# CONFIGURAÇÕES GLOBAIS DE TESTE E EXIBIÇÃO
# ==============================================================================
# 1. Índice do especialista para visualização/teste nas etapas intermediárias:
# 0: Dr. Marcos Vinícius Lima | 1: Dra. Beatriz | 2: Prof. Dr. Roberto | 3: Dr. Carlos | 4: Dra. Helena
indice = 2

# 2. Número global de casas decimais para exibição e arredondamento
decimais = 2

# Aplica a configuração no pandas
pd.set_option("display.float_format", f"{{:.{decimais}f}}".format)

# Função de arredondamento aritmético idêntica ao Excel (ROUND / ARRED)
# Regra do Excel: 5 ou mais arredonda para cima (afastando de zero), diferente do round bancário do Python
def arredondar_excel(valor, n_casas=None):
    if n_casas is None:
        n_casas = decimais
    if valor is None:
        return None
    if isinstance(valor, (int, float, np.floating, np.integer)):
        d = Decimal(str(valor))
        fator = Decimal("10") ** -n_casas
        return float(d.quantize(fator, rounding=ROUND_HALF_UP))
    elif isinstance(valor, list):
        return [arredondar_excel(v, n_casas) for v in valor]
    return valor

# Função de inversão termo a termo [1/l, 1/m, 1/u]
def inverter_fuzzy(triangular):
    l, m, u = triangular
    if l == 1 and m == 1 and u == 1:
        return [1.0, 1.0, 1.0]
    return [1.0 / l, 1.0 / m, 1.0 / u]


In [53]:
# ETAPA 1: Construção das Matrizes de Comparação Paritária (6x6)

fuzzy_data = []

for index, row in df.iterrows():
    # 1. Cria matriz 6x6 com [1, 1, 1] na diagonal principal
    matriz = []
    for i in range(n):
        linha = []
        for j in range(n):
            if i == j:
                linha.append([1.0, 1.0, 1.0])
            else:
                linha.append(None)
        matriz.append(linha)
    
    # 2. Preenche as comparações paritárias para cada par (i < j)
    for i in range(n):
        for j in range(i + 1, n):
            crit1 = criterios[i]
            crit2 = criterios[j]
            
            col_intensidade = f"Intensidade Saaty {crit1} Vs {crit2}"
            intensidade = row[col_intensidade]
            
            valor_direto = [float(x) for x in scale[intensidade]]
            valor_invertido = inverter_fuzzy(valor_direto)
            
            # Julgamento linha por coluna:
            # [i][j] (triângulo superior) recebe o valor invertido [1/l, 1/m, 1/u]
            # [j][i] (triângulo inferior) recebe o valor direto [l, m, u]
            matriz[i][j] = valor_invertido
            matriz[j][i] = valor_direto
                
    fuzzy_data.append({
        "Name": row["Name"],
        "Matrix": matriz
    })

print(f"Total de {len(fuzzy_data)} matrizes construídas com sucesso.")
print(f"Exibindo Matriz Paritária do Especialista Selecionado (índice {indice}: {fuzzy_data[indice]['Name']}):")

# Exibição da matriz com as listas internas arredondadas pelo sistema do Excel
df_matriz_exibicao = pd.DataFrame(
    [[arredondar_excel(celula) for celula in linha] for linha in fuzzy_data[indice]["Matrix"]],
    index=criterios,
    columns=criterios
)
display(df_matriz_exibicao)


Total de 5 matrizes construídas com sucesso.
Exibindo Matriz Paritária do Especialista Selecionado (índice 2: Prof. Dr. Roberto Guimarães):


,A,B,C,D,E,F
A,"[1.0, 1.0, 1.0]","[0.5, 0.33, 0.25]","[0.25, 0.2, 0.17]","[0.17, 0.14, 0.13]","[0.11, 0.11, 0.11]","[1.0, 1.0, 1.0]"
B,"[2.0, 3.0, 4.0]","[1.0, 1.0, 1.0]","[0.5, 0.33, 0.25]","[0.25, 0.2, 0.17]","[0.17, 0.14, 0.13]","[0.11, 0.11, 0.11]"
C,"[4.0, 5.0, 6.0]","[2.0, 3.0, 4.0]","[1.0, 1.0, 1.0]","[1.0, 1.0, 1.0]","[0.5, 0.33, 0.25]","[0.25, 0.2, 0.17]"
D,"[6.0, 7.0, 8.0]","[4.0, 5.0, 6.0]","[1.0, 1.0, 1.0]","[1.0, 1.0, 1.0]","[0.17, 0.14, 0.13]","[0.11, 0.11, 0.11]"
E,"[9.0, 9.0, 9.0]","[6.0, 7.0, 8.0]","[2.0, 3.0, 4.0]","[6.0, 7.0, 8.0]","[1.0, 1.0, 1.0]","[1.0, 1.0, 1.0]"
F,"[1.0, 1.0, 1.0]","[9.0, 9.0, 9.0]","[4.0, 5.0, 6.0]","[9.0, 9.0, 9.0]","[1.0, 1.0, 1.0]","[1.0, 1.0, 1.0]"


In [54]:
# ETAPA 2: Redução por linha via Média Geométrica Fuzzy (Geometric Mean)
# Para cada linha i, calcula-se a raiz n-ésima do produto de todas as colunas j

for item in fuzzy_data:
    matriz = item["Matrix"]
    geo_means = []
    
    for i in range(n):
        prod_l = 1.0
        prod_m = 1.0
        prod_u = 1.0
        
        for j in range(n):
            prod_l *= matriz[i][j][0]
            prod_m *= matriz[i][j][1]
            prod_u *= matriz[i][j][2]
            
        gm_l = prod_l ** (1.0 / n)
        gm_m = prod_m ** (1.0 / n)
        gm_u = prod_u ** (1.0 / n)
        
        geo_means.append([gm_l, gm_m, gm_u])
        
    item["Geometric_Mean"] = geo_means

# Exibição das Médias Geométricas do especialista selecionado (arredondamento Excel)
df_gm_exemplo = pd.DataFrame(
    [[arredondar_excel(x) for x in row] for row in fuzzy_data[indice]["Geometric_Mean"]],
    index=criterios,
    columns=pd.MultiIndex.from_product([["Geometric Mean"], ["F1", "F2", "F3"]])
)
print(f"Médias Geométricas (índice {indice}: {fuzzy_data[indice]['Name']}):")
display(df_gm_exemplo)


Médias Geométricas (índice 2: Prof. Dr. Roberto Guimarães):


Geometric Mean          
              F1   F2   F3
A           0.36 0.32 0.29
B           0.41 0.38 0.36
C           1.00 1.00 1.00
D           0.87 0.91 0.93
E           2.94 3.31 3.63
F           2.62 2.72 2.80

In [55]:
# ETAPA 3: Soma das Médias Geométricas (SUM) e Inverso da Soma (SUM^-1)
# SUM = soma de cada coluna de GeometricMean
# SUM^-1 = inverso de cada elemento da soma (1 / SUM)

for item in fuzzy_data:
    gm = item["Geometric_Mean"]
    
    sum_1 = sum(gm[i][0] for i in range(n))
    sum_2 = sum(gm[i][1] for i in range(n))
    sum_3 = sum(gm[i][2] for i in range(n))
    
    sum_inv = [1.0 / sum_1, 1.0 / sum_2, 1.0 / sum_3]
    
    item["SUM"] = [sum_1, sum_2, sum_3]
    item["SUM_INV"] = sum_inv

# Exibição de SUM e SUM^-1 para o especialista selecionado (arredondamento Excel)
sum_val = fuzzy_data[indice]["SUM"]
sum_inv_val = fuzzy_data[indice]["SUM_INV"]

df_sum = pd.DataFrame([
    [arredondar_excel(x) for x in sum_val],
    [arredondar_excel(x) for x in sum_inv_val]
], index=["SUM", "SUM^-1"], columns=["F1", "F2", "F3"])

print(f"Soma e Inverso SUM^-1 (índice {indice}: {fuzzy_data[indice]['Name']}):")
display(df_sum)


Soma e Inverso SUM^-1 (índice 2: Prof. Dr. Roberto Guimarães):


,F1,F2,F3
SUM,8.21,8.64,9.03
SUM^-1,0.12,0.12,0.11


In [56]:
# ETAPA 4: Cálculo dos Pesos Fuzzy (Fuzzy Weight - FW) usando max(SUM^-1, k)
# Conforme a fórmula do modelo:
# FW[crit, F1] = GeometricMean[crit, F1] * max(SUM^-1, 1)
# FW[crit, F2] = GeometricMean[crit, F2] * max(SUM^-1, 2)
# FW[crit, F3] = GeometricMean[crit, F3] * max(SUM^-1, 3)

for item in fuzzy_data:
    gm = item["Geometric_Mean"]
    sum_inv = item["SUM_INV"]
    
    # Fatores ordenados de forma decrescente: max(sum-1, k)
    fatores_ordenados = sorted(sum_inv, reverse=True)
    fator_1 = fatores_ordenados[0]  # max(sum-1, 1) = 1º maior
    fator_2 = fatores_ordenados[1]  # max(sum-1, 2) = 2º maior
    fator_3 = fatores_ordenados[2]  # max(sum-1, 3) = 3º maior
    
    item["FACTORS"] = [fator_1, fator_2, fator_3]
    
    fw = []
    for i in range(n):
        fw_1 = gm[i][0] * fator_1
        fw_2 = gm[i][1] * fator_2
        fw_3 = gm[i][2] * fator_3
        fw.append([fw_1, fw_2, fw_3])
        
    item["Fuzzy_Weight"] = fw

# Exibição dos fatores max(SUM^-1, k) e Pesos Fuzzy (FW) do especialista selecionado (arredondamento Excel)
fat_marcos = fuzzy_data[indice]["FACTORS"]
print(f"Fatores max(SUM^-1, k) (índice {indice}: {fuzzy_data[indice]['Name']}):")
print(f"  max(SUM^-1, 1) = {arredondar_excel(fat_marcos[0])}")
print(f"  max(SUM^-1, 2) = {arredondar_excel(fat_marcos[1])}")
print(f"  max(SUM^-1, 3) = {arredondar_excel(fat_marcos[2])}")

df_fw = pd.DataFrame(
    [[arredondar_excel(x) for x in row] for row in fuzzy_data[indice]["Fuzzy_Weight"]],
    index=criterios,
    columns=pd.MultiIndex.from_product([["Fuzzy Weight (FW)"], ["F1", "F2", "F3"]])
)
print(f"\nPesos Fuzzy FW (índice {indice}: {fuzzy_data[indice]['Name']}):")
display(df_fw)


Fatores max(SUM^-1, k) (índice 2: Prof. Dr. Roberto Guimarães):
  max(SUM^-1, 1) = 0.12
  max(SUM^-1, 2) = 0.12
  max(SUM^-1, 3) = 0.11

Pesos Fuzzy FW (índice 2: Prof. Dr. Roberto Guimarães):


Fuzzy Weight (FW)          
                 F1   F2   F3
A              0.04 0.04 0.03
B              0.05 0.04 0.04
C              0.12 0.12 0.11
D              0.11 0.10 0.10
E              0.36 0.38 0.40
F              0.32 0.31 0.31

In [57]:
# ETAPA 5: Defuzzificação (Mean(FW)) e Vetor de Prioridades Normalizado (W)
# Mean(FW) = média aritmética dos 3 componentes: (FW1 + FW2 + FW3) / 3
# W = Mean(FW) normalizado para somar 1.0

for item in fuzzy_data:
    fw = item["Fuzzy_Weight"]
    
    mean_fw = [(fw[i][0] + fw[i][1] + fw[i][2]) / 3.0 for i in range(n)]
    soma_mean_fw = sum(mean_fw)
    W = [m_fw / soma_mean_fw for m_fw in mean_fw]
    
    item["Mean_FW"] = mean_fw
    item["W"] = W

# Exibição de Mean(FW) e W para o especialista selecionado (arredondamento Excel)
df_w = pd.DataFrame({
    "Mean(FW)": [arredondar_excel(x) for x in fuzzy_data[indice]["Mean_FW"]],
    "W": [arredondar_excel(x) for x in fuzzy_data[indice]["W"]]
}, index=criterios)

print(f"Pesos Defuzzificados Mean(FW) e Vetor de Prioridades W (índice {indice}: {fuzzy_data[indice]['Name']}):")
display(df_w)
print(f"Soma de W = {arredondar_excel(sum(fuzzy_data[indice]['W']))}")


Pesos Defuzzificados Mean(FW) e Vetor de Prioridades W (índice 2: Prof. Dr. Roberto Guimarães):


,Mean(FW),W
A,0.04,0.04
B,0.04,0.04
C,0.12,0.12
D,0.10,0.10
E,0.38,0.38
F,0.31,0.31


Soma de W = 1.0


In [58]:
# ETAPA 6: Classificação / Ranking dos Critérios (1 = mais prioritário)

for item in fuzzy_data:
    w_series = pd.Series(item["W"], index=criterios)
    item["Ranking"] = w_series.rank(ascending=False, method="min").astype(int)

# Exibição do Ranking do especialista selecionado (arredondamento Excel)
df_rank = pd.DataFrame({
    "Critério": criterios,
    "W": [arredondar_excel(x) for x in fuzzy_data[indice]["W"]],
    "Rank": fuzzy_data[indice]["Ranking"].values
}).sort_values("Rank").reset_index(drop=True)

print(f"Ranking dos Critérios (índice {indice}: {fuzzy_data[indice]['Name']}):")
display(df_rank)


Ranking dos Critérios (índice 2: Prof. Dr. Roberto Guimarães):


,Critério,W,Rank
0,E,0.38,1
1,F,0.31,2
2,C,0.12,3
3,D,0.10,4
4,B,0.04,5
5,A,0.04,6


In [59]:
# ETAPA 7: PRODUTO FINAL - Vetor de Prioridades W para todas as pessoas do BD

linhas_finais = []
for item in fuzzy_data:
    registro = {"Especialista": item["Name"]}
    for crit, peso in zip(criterios, item["W"]):
        registro[crit] = arredondar_excel(peso)
    registro["Soma"] = arredondar_excel(sum(item["W"]))
    linhas_finais.append(registro)

df_final_W = pd.DataFrame(linhas_finais).set_index("Especialista")

print("=" * 80)
print("PRODUTO FINAL: VETOR DE PRIORIDADES W CONSOLIDADO (TODOS OS ESPECIALISTAS DO BD)")
print("=" * 80)
display(df_final_W)


PRODUTO FINAL: VETOR DE PRIORIDADES W CONSOLIDADO (TODOS OS ESPECIALISTAS DO BD)


,A,B,C,D,E,F,Soma
Especialista,,,,,,,
Dr. Marcos Vinícius Lima,0.04,0.05,0.08,0.14,0.18,0.52,1.00
Dra. Beatriz Albuquerque,0.04,0.07,0.10,0.20,0.16,0.44,1.00
Prof. Dr. Roberto Guimarães,0.04,0.04,0.12,0.10,0.38,0.31,1.00
Dr. Carlos Eduardo Mendes,0.04,0.08,0.09,0.13,0.24,0.41,1.00
Dra. Helena Vasconcelos,0.04,0.07,0.11,0.22,0.30,0.26,1.00


In [60]:
# ETAPA 8: Visão Detalhada Completa (idêntica ao modelo da planilha do Excel)

def exibir_detalhamento_excel(indice_esp):
    item = fuzzy_data[indice_esp]
    nome = item["Name"]
    
    colunas = pd.MultiIndex.from_tuples([
        ("Geometric Mean", "F1"), ("Geometric Mean", "F2"), ("Geometric Mean", "F3"),
        ("Fuzzy Weight (FW)", "F1"), ("Fuzzy Weight (FW)", "F2"), ("Fuzzy Weight (FW)", "F3"),
        ("Mean(FW)", ""), ("W", ""), ("Rank", "")
    ])
    
    linhas = []
    for i in range(n):
        linhas.append([
            arredondar_excel(item["Geometric_Mean"][i][0]),
            arredondar_excel(item["Geometric_Mean"][i][1]),
            arredondar_excel(item["Geometric_Mean"][i][2]),
            arredondar_excel(item["Fuzzy_Weight"][i][0]),
            arredondar_excel(item["Fuzzy_Weight"][i][1]),
            arredondar_excel(item["Fuzzy_Weight"][i][2]),
            arredondar_excel(item["Mean_FW"][i]),
            arredondar_excel(item["W"][i]),
            item["Ranking"].iloc[i]
        ])
        
    df_det = pd.DataFrame(linhas, index=criterios, columns=colunas)
    print(f"\n=======================================================")
    print(f"Detalhamento Completo F-AHP: {nome} (índice {indice_esp})")
    print(f"=======================================================")
    display(df_det)

# Exibição para o especialista selecionado (variável indice)
exibir_detalhamento_excel(indice)



Detalhamento Completo F-AHP: Prof. Dr. Roberto Guimarães (índice 2)


Geometric Mean           Fuzzy Weight (FW)           Mean(FW)    W Rank
              F1   F2   F3                F1   F2   F3                   
A           0.36 0.32 0.29              0.04 0.04 0.03     0.04 0.04    6
B           0.41 0.38 0.36              0.05 0.04 0.04     0.04 0.04    5
C           1.00 1.00 1.00              0.12 0.12 0.11     0.12 0.12    3
D           0.87 0.91 0.93              0.11 0.10 0.10     0.10 0.10    4
E           2.94 3.31 3.63              0.36 0.38 0.40     0.38 0.38    1
F           2.62 2.72 2.80              0.32 0.31 0.31     0.31 0.31    2

In [61]:
# ETAPA 9: Vetor de Prioridades Médio Global (Média de Todos os Usuários)
# Calcula a média dos pesos W de cada critério para todos os especialistas do banco de dados,
# gerando um único vetor de prioridades global representativo do grupo.

# Média aritmética dos pesos de cada critério através de todos os especialistas
w_global = df_final_W[criterios].mean()

# 1. Vetor de Prioridades Global (Formato de Linha Única com arredondamento Excel)
df_vetor_global = pd.DataFrame(
    [[arredondar_excel(w_global[crit]) for crit in criterios]],
    index=["W_Global"],
    columns=criterios
)
df_vetor_global["Soma"] = arredondar_excel(w_global.sum())

print("=" * 80)
print("VETOR DE PRIORIDADES MÉDIO GLOBAL (CONSOLIDADO DE TODOS OS USUÁRIOS)")
print("=" * 80)
display(df_vetor_global)

# 2. Ranking Global dos Critérios
df_ranking_global = (
    pd.DataFrame({
        "Critério": criterios,
        "Peso Médio (%)": [arredondar_excel(x * 100) for x in w_global.values],
        "Rank": (-w_global).rank(method="min").astype(int).values,
    })
    .sort_values("Rank")
    .reset_index(drop=True)
)

print("\nRanking Global dos Critérios:")
display(df_ranking_global)


VETOR DE PRIORIDADES MÉDIO GLOBAL (CONSOLIDADO DE TODOS OS USUÁRIOS)


,A,B,C,D,E,F,Soma
W_Global,0.04,0.06,0.10,0.16,0.25,0.39,1.00



Ranking Global dos Critérios:


,Critério,Peso Médio (%),Rank
0,F,38.80,1
1,E,25.20,2
2,D,15.80,3
3,C,10.00,4
4,B,6.20,5
5,A,4.00,6
